# VTM parcial no Taskonomy

Notebook fino sobre o pacote `vtm`. Ative uma GPU em **Runtime → Change runtime type** antes do treino.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

A célula seguinte clona este repositório do GitHub na primeira execução e executa `git pull` nas sessões seguintes.

In [ ]:
from pathlib import Path
import subprocess

repo = Path('/content/Visual-Token-Matching')
if not (repo / '.git').exists():
    subprocess.run([
        'git', 'clone',
        'https://github.com/NataLira1/Visual-Token-Matching.git',
        str(repo),
    ], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)

%cd /content/Visual-Token-Matching
!pip install -q -e ".[experiment]"
import torch, timm
print('torch', torch.__version__, 'timm', timm.__version__, 'gpu', torch.cuda.get_device_name(0))

## Download seletivo

Esta etapa corrige uma incompatibilidade conhecida do `omnidata-tools==0.0.23`, normaliza automaticamente espaços/acentos do nome exigido pelo downloader, valida o e-mail e interrompe o notebook se o dry-run não chegar à confirmação da licença. O download e o pré-processamento não fazem parte das quatro horas de GPU.

In [ ]:
!sudo apt-get -qq update && sudo apt-get -qq install -y aria2
import inspect
import re
import subprocess
import unicodedata
from pathlib import Path

# omnidata-tools 0.0.23 adiciona o componente interno "omnidata" à
# verificação, mas esquece sua entrada em STARTER_DATA_LICENSES. Isso causa
# KeyError: 'omnidata' logo após imprimir a licença do Taskonomy. A correção
# fica restrita ao pacote temporário instalado neste runtime do Colab.
import omnidata_tools.starter_dataset as starter_dataset

license_module_path = Path(inspect.getfile(starter_dataset))
compatibility_marker = '# vtm-colab-omnidata-license-compatibility'
if 'omnidata' not in starter_dataset.STARTER_DATA_LICENSES:
    module_source = license_module_path.read_text(encoding='utf-8')
    if compatibility_marker not in module_source:
        with license_module_path.open('a', encoding='utf-8') as module_file:
            module_file.write(
                "\n" + compatibility_marker + "\n"
                "STARTER_DATA_LICENSES.setdefault(\n"
                "    'omnidata', 'https://creativecommons.org/licenses/by/4.0/legalcode'\n"
                ")\n"
            )
    starter_dataset.STARTER_DATA_LICENSES.setdefault(
        'omnidata', 'https://creativecommons.org/licenses/by/4.0/legalcode'
    )
    print('Compatibilidade do omnidata-tools 0.0.23 aplicada.')
else:
    print('Mapa de licenças do omnidata-tools já está completo.')

license_check = subprocess.run(
    [
        'python', '-c',
        "from omnidata_tools.starter_dataset import STARTER_DATA_LICENSES; "
        "assert 'omnidata' in STARTER_DATA_LICENSES",
    ],
    text=True,
    capture_output=True,
)
if license_check.returncode != 0:
    raise RuntimeError(
        'Não foi possível corrigir o mapa de licenças do omnidata-tools. '
        + license_check.stderr
    )

raw_download_name = input('Nome completo para aceitar os termos do dataset: ').strip()
download_email = input('E-mail válido: ').strip()

if not raw_download_name:
    raise ValueError('O nome não pode ficar vazio.')
if not re.fullmatch(r'[^@\s]+@[^@\s]+\.[^@\s]+', download_email):
    raise ValueError('Informe um e-mail válido.')

# omnidata-tools 0.0.23 não faz URL-encode do nome antes de enviá-lo.
download_name = (
    unicodedata.normalize('NFKD', raw_download_name)
    .encode('ascii', 'ignore')
    .decode('ascii')
)
download_name = re.sub(r'[^A-Za-z0-9._-]+', '-', download_name).strip('-')
if not download_name:
    raise ValueError('Não foi possível normalizar o nome informado.')
print('Nome normalizado enviado ao downloader:', download_name)

download_command = [
    'omnitools.download', 'rgb', 'segment_semantic',
    '--components', 'taskonomy',
    '--subset', 'tiny',
    '--dest', '/content/drive/MyDrive/taskonomy',
    '--connections_total', '16',
    '--agree_all',
    '--name', download_name,
    '--email', download_email,
]

dry_run = subprocess.run(
    download_command + ['--dryrun'],
    text=True,
    capture_output=True,
)
safe_stdout = dry_run.stdout.replace(download_email, '[e-mail ocultado]')
safe_stderr = dry_run.stderr.replace(download_email, '[e-mail ocultado]')
print(safe_stdout)
if safe_stderr:
    print('--- stderr do downloader ---')
    print(safe_stderr)
print('Código de saída do dry-run:', dry_run.returncode)

if dry_run.returncode != 0:
    raise RuntimeError(
        'O dry-run falhou. Não execute a célula de download real. '
        'Consulte o stderr exibido acima.'
    )
if "Confirmation supplied by option '--agree_all'" not in dry_run.stdout:
    raise RuntimeError(
        'O comando terminou sem confirmar a licença. '
        'Não execute a célula de download real.'
    )
dry_run_ok = True
print('Dry-run validado. A próxima célula pode ser executada.')

In [ ]:
from pathlib import Path

if not globals().get('dry_run_ok', False):
    raise RuntimeError('Execute e valide primeiro a célula de dry-run.')

download_result = subprocess.run(download_command)
if download_result.returncode != 0:
    raise RuntimeError(
        f'Download falhou com código {download_result.returncode}. '
        'Não avance para a preparação do manifest.'
    )

data_root = Path('/content/drive/MyDrive/taskonomy')
downloaded_files = [path for path in data_root.rglob('*') if path.is_file()]
if not downloaded_files:
    raise RuntimeError(
        'O downloader retornou sucesso, mas nenhum arquivo foi encontrado no destino.'
    )
print(f'Download validado: {len(downloaded_files)} arquivos encontrados.')
for path in downloaded_files[:10]:
    print(path)

## Preparação, meta-treino e avaliação

In [ ]:
from pathlib import Path
import yaml
config = yaml.safe_load(Path('configs/taskonomy_vtm.yaml').read_text())
config['data']['root'] = '/content/drive/MyDrive/taskonomy'
config['data']['manifest'] = '/content/drive/MyDrive/vtm_outputs/taskonomy_manifest.json'
config['train']['checkpoint'] = '/content/drive/MyDrive/vtm_outputs/vtm_best.pt'
config['experiment']['output_dir'] = '/content/drive/MyDrive/vtm_outputs/evaluation'
runtime_config = Path('/content/vtm_taskonomy_runtime.yaml')
runtime_config.write_text(yaml.safe_dump(config, sort_keys=False))
runtime_config

In [ ]:
!vtm-taskonomy prepare --config /content/vtm_taskonomy_runtime.yaml
!vtm-taskonomy train --config /content/vtm_taskonomy_runtime.yaml
!vtm-taskonomy evaluate --config /content/vtm_taskonomy_runtime.yaml

In [ ]:
import pandas as pd
from IPython.display import display, Image
out = Path(config['experiment']['output_dir'])
display(pd.read_csv(out / 'summary.csv'))
display(pd.read_json(out / 'hypothesis.json'))
panels = sorted((out / 'panels').glob('*.png'))
if panels:
    display(Image(filename=str(panels[0])))